## NLP Final
### Part 3: Named Entity Recognition
### Aren Mizuno
### March 12, 2026

In [1]:
# Imports
!pip -q install -U spacy pyarrow pandas tqdm scikit-learn
!python -m spacy download en_core_web_lg -q

import re
import pandas as pd
import spacy
import torch

from tqdm.auto import tqdm
from google.colab import drive
from spacy.matcher import PhraseMatcher


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 133.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 68.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.1 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.1 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.1 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0

In [2]:
# Cuda
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [3]:
# Mount drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
# Load the cleaned dataset
data_path = "/content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/parquets/news_final_project_cleaned.parquet"
df = pd.read_parquet(data_path, engine="pyarrow")
df.head(2)

,url,date,title,text
0,https://blockworks.co/price/bad,2025-06-23,"Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod..."
1,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,This AI video of gymnastics might be the freak...,This AI video of gymnastics might be the freak...


In [5]:
# Text cleaning helpers
def clean_match_text(s):
    s = str(s).lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s


def clean_article_text(text):
    text = str(text)

    # Remove URLs
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    # Remove common website boilerplate
    boilerplate_patterns = [
        r"\bopen menu\b",
        r"\bmenu\b",
        r"\bsearch\b",
        r"\blogin\b",
        r"\blog in\b",
        r"\bsign up\b",
        r"\bsign in\b",
        r"\bsubscribe\b",
        r"\bprivacy policy\b",
        r"\bterms of service\b",
        r"\bterms and conditions\b",
        r"\bcontact us\b",
        r"\bcontact support\b",
        r"\bnewsletter\b",
        r"\bskip to content\b",
        r"\bhome\b",
        r"\bworld\b",
        r"\bbusiness\b",
        r"\bfinance\b",
        r"\bsports\b",
        r"\bweather\b",
        r"\bentertainment\b",
        r"\bstore\b",
        r"\bblog\b",
        r"\bforums\b",
        r"\bshop\b",
        r"\bfollow us\b",
        r"\bread the rules\b",
        r"\bmobile app\b",
        r"\bmail\b",
        r"\byahoo finance\b",
        r"\bmore\b",
    ]

    for pat in boilerplate_patterns:
        text = re.sub(pat, " ", text, flags=re.IGNORECASE)

    # Remove time stamps like 23:08:29 PKT
    text = re.sub(r"\b\d{1,2}:\d{2}:\d{2}\s*[A-Z]{2,5}\b", " ", text)

    # Remove short ticker fragments like [BFRG]
    text = re.sub(r"\[[A-Z0-9\.\-]{1,10}\]", " ", text)

    # Remove weird concatenated site-chrome tokens
    text = re.sub(r"\b[A-Za-z]+(?:[A-Z][a-z]+){2,}\b", " ", text)

    # Remove separator junk
    text = re.sub(r"[_|]+", " ", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


df["text"] = df["text"].fillna("").astype(str)
df["text_for_extraction"] = df["text"].map(clean_article_text)
df["text_clean"] = df["text_for_extraction"].map(clean_match_text)

In [6]:
# Load model
nlp_ner = spacy.load(
    "en_core_web_lg",
    disable=["tagger", "parser", "lemmatizer", "attribute_ruler"]
)

nlp_match = spacy.blank("en")

In [7]:
# Technology extraction
TECH_TERMS = [
    "artificial intelligence",
    "machine learning",
    "deep learning",
    "generative ai",
    "genai",
    "large language model",
    "large language models",
    "foundation model",
    "foundation models",
    "natural language processing",
    "computer vision",
    "retrieval augmented generation",
    "retrieval-augmented generation",
    "rag",
    "prompt engineering",
    "fine tuning",
    "fine-tuning",
    "vector database",
    "vector databases",
    "embedding model",
    "embedding models",
    "embeddings",
    "neural network",
    "neural networks",
    "transformer",
    "transformer model",
    "transformer models",
    "supervised learning",
    "unsupervised learning",
    "reinforcement learning",
    "multimodal ai",
    "language model",
    "language models",
    "ai model",
    "ai models",
    "chatbot",
    "chatbots",
    "autonomous driving",
    "robotics",
]

TECH_CANONICAL_MAP = {
    "rag": "retrieval-augmented generation",
    "retrieval augmented generation": "retrieval-augmented generation",
    "llm": "large language model",
    "llms": "large language model",
    "large language models": "large language model",
    "foundation models": "foundation model",
    "neural networks": "neural network",
    "vector databases": "vector database",
    "embedding models": "embedding model",
    "fine tuning": "fine-tuning",
    "genai": "generative ai",
    "language model": "large language model",
    "language models": "large language model",
    "transformer models": "transformer model",
    "ai models": "ai model",
    "chatbots": "chatbot",
}

TECH_TERMS = sorted(set(TECH_TERMS), key=len, reverse=True)

matcher = PhraseMatcher(nlp_match.vocab, attr="LOWER")
patterns = [nlp_match.make_doc(term) for term in TECH_TERMS]
matcher.add("TECH", patterns)


def normalize_phrase(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = text.strip(" -–—.,;:()[]{}\"'")
    return text


def canonicalize_tech(term):
    term = normalize_phrase(term)
    return TECH_CANONICAL_MAP.get(term, term)


def extract_technologies_fast(text):
    doc = nlp_match.make_doc(text)
    tech = set()

    # Exact phrase matches
    for _, start, end in matcher(doc):
        span = doc[start:end].text
        tech.add(canonicalize_tech(span))

    # Regex aliases
    regex_aliases = {
        r"\bllm\b": "large language model",
        r"\bllms\b": "large language model",
        r"\brag\b": "retrieval-augmented generation",
        r"\bgenai\b": "generative ai",
        r"\blanguage models?\b": "large language model",
        r"\bai models?\b": "ai model",
    }

    for pat, replacement in regex_aliases.items():
        for _ in re.finditer(pat, text, flags=re.IGNORECASE):
            tech.add(replacement)

    return sorted(tech)


In [8]:
# Company extraction rules
BAD_COMPANIES = {
    "ai", "news", "media", "company", "group", "press", "staff", "editor",
    "business", "finance", "home", "world", "sports", "weather", "menu",
    "search", "login", "subscribe", "newsletter", "content", "comments",
    "privacy", "policy", "terms", "service", "services"
}

BAD_COMPANY_WORDS = {
    "news", "video", "article", "analysis", "comment", "comments",
    "subscribe", "weather", "sports", "world", "finance", "business",
    "login", "menu", "search", "policy", "terms", "service",
    "services", "support", "contact", "magazine", "blog", "blogs",
    "newsletter", "follow", "latest", "stories", "story", "read",
    "watch", "content", "podcast", "calculator", "shop", "store"
}

BAD_PUBLISHERS = {
    "Reuters", "Boing Boing", "FedTech", "Economic Times",
    "The Economic Times", "Forbes", "Bloomberg", "CNN", "BBC",
    "ExBulletin", "The DBT News", "Crooks and Liars", "Yahoo Finance",
    "The A.V. Club", "Gizmodo", "Quartz", "Kotaku", "Lifehacker",
    "TechCrunch", "Wired", "MarketWatch", "Business Insider",
    "Associated Press", "AP"
}

GOOD_SINGLE_WORD_COMPANIES = {
    "Google", "Microsoft", "Apple", "Amazon", "Meta", "OpenAI",
    "Anthropic", "Nvidia", "Tesla", "Qualcomm", "Zoom", "Baidu",
    "Intel", "AMD", "Oracle", "IBM", "Adobe", "Solana",
    "Salesforce", "Cisco", "Samsung", "Alphabet", "Hasbro",
    "Chegg", "Mistral", "Huawei", "Instagram", "Databricks",
    "Snowflake", "Palantir", "Alibaba", "Dell", "Arm"
}

LEGAL_SUFFIXES = {
    "inc", "corp", "corporation", "ltd", "llc", "plc", "holdings", "group"
}

BAD_ENTITY_PATTERNS = [
    r"privacy policy",
    r"terms of service",
    r"terms and conditions",
    r"contact us",
    r"contact support",
    r"sign in",
    r"sign up",
    r"skip to content",
    r"open menu",
    r"follow us",
    r"read the rules",
]

PRODUCT_WORDS = {
    "studio", "api", "search", "image", "creator", "companion",
    "windows", "iphone", "ipad", "chatgpt", "bard", "gemini",
    "copilot", "edge", "bing", "dall-e", "phi", "ultra",
    "assistant", "meetings", "meeting", "phone", "creator"
}


def normalize_company(text):
    text = str(text).strip()
    text = re.sub(r"[’']s\b", "", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip(" -–—.,;:()[]{}\"'`?!&")
    return text


def clean_company(name):
    name = normalize_company(name)

    if not name:
        return None

    low = name.lower()

    if low in BAD_COMPANIES:
        return None

    if name in BAD_PUBLISHERS:
        return None

    if len(name) <= 2:
        return None

    if not re.search(r"[A-Za-z]", name):
        return None

    for pat in BAD_ENTITY_PATTERNS:
        if re.search(pat, low):
            return None

    if ".com" in low or ".net" in low or ".org" in low or "http" in low or "www" in low:
        return None

    if len(name.split()) > 4:
        return None

    words = set(re.findall(r"[A-Za-z0-9'\-]+", low))
    if words & BAD_COMPANY_WORDS:
        return None

    if any(w in PRODUCT_WORDS for w in words) and name not in GOOD_SINGLE_WORD_COMPANIES:
        return None

    if len(name.split()) == 1:
        if name not in GOOD_SINGLE_WORD_COMPANIES:
            if len(name) < 5:
                return None
            if not re.match(r"^[A-Z][A-Za-z0-9&\-.]*$", name):
                return None
    else:
        alpha_words = [w for w in name.split() if re.search(r"[A-Za-z]", w)]
        if not alpha_words:
            return None
        if not all(w[0].isupper() or w.isupper() for w in alpha_words):
            return None

    return name


def dedupe_companies(companies):
    companies = sorted(set(companies), key=lambda x: (-len(x), x))
    kept = []

    for c in companies:
        c_low = c.lower()
        if any(c_low in existing.lower() for existing in kept):
            continue
        kept.append(c)

    return sorted(kept)


def count_mentions(text, company):
    return len(re.findall(re.escape(company), text, flags=re.IGNORECASE))


def score_company(company, text):
    score = 0

    mentions = count_mentions(text, company)
    score += min(mentions, 3)

    if company in GOOD_SINGLE_WORD_COMPANIES:
        score += 2

    low_words = set(re.findall(r"[A-Za-z0-9'\-]+", company.lower()))
    if low_words & LEGAL_SUFFIXES:
        score += 2

    if len(company.split()) >= 4:
        score -= 1

    return score


def extract_companies_from_doc(doc, text, top_k=3):
    candidates = set()

    for ent in doc.ents:
        if ent.label_ == "ORG":
            c = clean_company(ent.text)
            if c:
                candidates.add(c)

    candidates = dedupe_companies(candidates)

    scored = [(c, score_company(c, text)) for c in candidates]
    scored = [x for x in scored if x[1] >= 2]
    scored = sorted(scored, key=lambda x: (-x[1], x[0]))

    return [c for c, _ in scored[:top_k]]

In [9]:
# Company extraction
texts = df["text_for_extraction"].tolist()

companies_all = []

for text, doc in tqdm(
    zip(texts, nlp_ner.pipe(texts, batch_size=128)),
    total=len(df),
    desc="Extracting companies"
):
    companies = extract_companies_from_doc(doc, text, top_k=3)
    companies_all.append(companies)

df["company"] = companies_all

Extracting companies:   0%|          | 0/136233 [00:00<?, ?it/s]

In [10]:
# Technology extraction
df["technology"] = [
    extract_technologies_fast(text)
    for text in tqdm(df["text_clean"], total=len(df), desc="Extracting technologies")
]

Extracting technologies:   0%|          | 0/136233 [00:00<?, ?it/s]

In [11]:
# Display
with pd.option_context("display.max_colwidth",200):
    display(df[["text","company","technology"]].head(20))

,text,company,technology
0,"Bad Idea AI Price (BAD), Market Cap, Price Today & Chart History - BlockworksOpen menuBrandsnewsletterspodcastseventsroundtablestoken transparencyetf trackerpricesresearchanalyticshomepricesBad Id...",[Bad Idea AI Price],[]
1,This AI video of gymnastics might be the freakiest I've seen yet - Boing Boing MENU SEARCH STORE MENU SEARCH STORE Blog : The posts Forums : Read the rules Store : Wonderful Products (Contact Supp...,"[Instagram, Trump, Werners AI Art]",[]
2,"If using AI feels like a chore, try this - Boing Boing MENU SEARCH STORE MENU SEARCH STORE Blog : The posts Forums : Read the rules Store : Wonderful Products (Contact Support) Newsletter : Daily ...","[Apple, Lifetime Subscription, Frenchie]","[ai model, fine-tuning]"
3,The Road Ahead: How China's AI Foundation Model is Shaping the Future of Autonomous Driving Technology Ir ao contidoVen. 10 de novembro de 2023 Vida da cidadePresentando as novas tecnoloxías e o p...,"[Baidu, The AI Foundation Model]","[artificial intelligence, autonomous driving, deep learning, foundation model]"
4,"Microsoft and Nvidia to Empower Developers with Windows AI Studio мазмұнға өтуСенбі. 18 қараша, 2023 жыл Қала өміріЖаңа технологиялар мен AI күшін ашу AIжаңалықтарғарыштехнологиясерікғылымАҚШбайла...","[Microsoft, Mistral, Nvidia]","[ai model, large language model]"
5,Google Releases New Chatbot Bard as a Strong Competitor to OpenAI's ChatGPT Wiessel un InhaltDi. 12 Dezember 2023 Stad LiewenEntdeckt nei Technologien an d'Kraaft vun AI AINeiegkeetenSpaceTechnolo...,[],"[chatbot, large language model]"
6,"Zoom Expands AI Offering with AI Companion and Revenue Accelerator Skip to contentThu. Sep 7th, 2023 CityLifeUnveiling New Technologies and the Power of AI AINewsSpaceTechnologySatelliteScienceU.S...","[Google, Microsoft, Zoom AI]","[artificial intelligence, generative ai]"
7,"Pro-AI Thinking: Enhancing Industrial Environments with Artificial Intelligence Skip to content Fri. Aug 4th, 2023 CityLife The Power of AI Models AI Drones Military News Satellite Telephony Satel...","[Pro-AI, Revolutionizing Office Management]","[ai model, artificial intelligence, robotics]"
8,Best AI Prompts for Business Risk ManagementProductSolutionsLearnPricingEnterpriseContact SalesSign UpLog inMenuClose MenuProductAI built for youTasksDocsGoalsWhiteboardsDashboardsChatPlatformRapi...,"[ClickUp AI, AI Tools]",[machine learning]
9,State AGs Warn AI Companies: Clean Up Your Child Safeguards | Crooks and Liars Support C&L — Go Ad-Free Today Ad Free Login CLTV Follow us on Bluesky Like us on Facebook Follow us on Twitter Follo...,"[Facebook, Microsoft, Twitter]","[artificial intelligence, generative ai]"


In [12]:
# Save as parquet
save_path_parquet = "/content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/parquets/ner.parquet"

df.to_parquet(save_path_parquet, engine="pyarrow", index=False)

print("Saved parquet to:", save_path_parquet)

Saved parquet to: /content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/parquets/ner.parquet
